## Activity 1 — Text Classification with RNN

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import tiktoken

from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("IMDB Dataset.csv")

print(df.head())
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [4]:
df = df.sample(n=3000, random_state=42)

print("Dataset size:", df.shape)

Dataset size: (3000, 2)


In [5]:
df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df.head())

                                                  review  sentiment
33553  I really liked this Summerslam due to the look...          1
9427   Not many television shows appeal to quite as m...          1
199    The film quickly gets to a major chase scene w...          0
12447  Jane Austen would definitely approve of this o...          1
39489  Expectations were somewhat high for me when I ...          0


In [6]:
reviews = df["review"].tolist()
labels = df["sentiment"].tolist()

print("Number of reviews:", len(reviews))
print("Number of labels:", len(labels))

Number of reviews: 3000
Number of labels: 3000


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    reviews,
    labels,
    test_size=0.2,
    random_state=42
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 2400
Testing reviews: 600


In [8]:
tokenizer = tiktoken.get_encoding("cl100k_base")

print("Vocabulary size:", tokenizer.n_vocab)

Vocabulary size: 100277


In [9]:
def tokenize_text(text):
    tokens = tokenizer.encode(text)

    # Add 1 so that 0 can be used for padding
    tokens = [token + 1 for token in tokens]

    return tokens

In [10]:
X_train_tokens = [tokenize_text(text) for text in X_train]
X_test_tokens = [tokenize_text(text) for text in X_test]

print(X_train_tokens[0][:20])

[73, 1098, 265, 2467, 8572, 316, 24319, 820, 1765, 82470, 324, 25429, 17873, 14, 359, 15747, 421, 4633, 1307, 603]


In [11]:
MAX_LEN = 100

def pad_sequence(tokens, max_len):

    # Truncate if the review is too long
    if len(tokens) > max_len:
        tokens = tokens[:max_len]

    # Pad with 0 if the review is too short
    while len(tokens) < max_len:
        tokens.append(0)

    return tokens

In [12]:
X_train_padded = [
    pad_sequence(tokens, MAX_LEN)
    for tokens in X_train_tokens
]

X_test_padded = [
    pad_sequence(tokens, MAX_LEN)
    for tokens in X_test_tokens
]

print("Length of one review:", len(X_train_padded[0]))

Length of one review: 100


In [13]:
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print("X_train shape:", X_train_tensor.shape)
print("y_train shape:", y_train_tensor.shape)

X_train shape: torch.Size([2400, 100])
y_train shape: torch.Size([2400])


In [14]:
class RNNClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # RNN layer
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        # Output layer
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        # Convert token IDs into embeddings
        embedded = self.embedding(x)

        # Pass embeddings through RNN
        output, hidden = self.rnn(embedded)

        # Take the last output
        last_output = output[:, -1, :]

        # Final prediction
        output = self.fc(last_output)

        return output

In [15]:
vocab_size = tokenizer.n_vocab + 1
embedding_dim = 64
hidden_dim = 64

model = RNNClassifier(
    vocab_size,
    embedding_dim,
    hidden_dim
)

print(model)

RNNClassifier(
  (embedding): Embedding(100278, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [16]:
loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [17]:
epochs = 5

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    # Forward pass
    outputs = model(X_train_tensor)

    # Remove extra dimension
    outputs = outputs.squeeze()

    # Calculate loss
    loss = loss_function(
        outputs,
        y_train_tensor
    )

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    print(
        "Epoch:",
        epoch + 1,
        "Loss:",
        loss.item()
    )

Epoch: 1 Loss: 0.7103135585784912
Epoch: 2 Loss: 0.7049849629402161
Epoch: 3 Loss: 0.70014488697052
Epoch: 4 Loss: 0.6957376599311829
Epoch: 5 Loss: 0.6917126178741455


In [18]:
model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_outputs = test_outputs.squeeze()

    predictions = (torch.sigmoid(test_outputs) >= 0.5).float()

accuracy = (predictions == y_test_tensor).float().mean()

print("Test Accuracy:", accuracy.item() * 100, "%")

Test Accuracy: 49.000000953674316 %


In [19]:
new_reviews = [
    "This movie was absolutely fantastic. The acting and story were excellent.",
    "I really disliked this movie. It was boring and the story was terrible.",
    "The movie was enjoyable and had some great performances."
]

In [20]:
new_reviews_tokens = [
    tokenize_text(review)
    for review in new_reviews
]

new_reviews_padded = [
    pad_sequence(tokens, MAX_LEN)
    for tokens in new_reviews_tokens
]

new_reviews_tensor = torch.tensor(
    new_reviews_padded,
    dtype=torch.long
)

In [21]:
model.eval()

with torch.no_grad():
    outputs = model(new_reviews_tensor)
    probabilities = torch.sigmoid(outputs).squeeze()

for review, probability in zip(new_reviews, probabilities):

    if probability >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    print("Review:", review)
    print("Predicted Sentiment:", sentiment)
    print("Probability:", probability.item())
    print("-" * 60)

Review: This movie was absolutely fantastic. The acting and story were excellent.
Predicted Sentiment: Positive
Probability: 0.5197518467903137
------------------------------------------------------------
Review: I really disliked this movie. It was boring and the story was terrible.
Predicted Sentiment: Positive
Probability: 0.5197518467903137
------------------------------------------------------------
Review: The movie was enjoyable and had some great performances.
Predicted Sentiment: Positive
Probability: 0.5197518467903137
------------------------------------------------------------


## Activity 2 — Text Classification with LSTM

In [22]:
import torch
import torch.nn as nn

In [23]:
class LSTMClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # LSTM layer
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        # Linear output layer
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        # Convert token IDs into embeddings
        embedded = self.embedding(x)

        # Pass embeddings through LSTM
        output, (hidden, cell) = self.lstm(embedded)

        # Take the last output
        last_output = output[:, -1, :]

        # Final prediction
        output = self.fc(last_output)

        return output

In [24]:
vocab_size = tokenizer.n_vocab + 1

embedding_dim = 64
hidden_dim = 64

print("Vocabulary size:", vocab_size)
print("Embedding dimension:", embedding_dim)
print("Hidden dimension:", hidden_dim)

Vocabulary size: 100278
Embedding dimension: 64
Hidden dimension: 64


In [25]:
lstm_model = LSTMClassifier(
    vocab_size,
    embedding_dim,
    hidden_dim
)

print(lstm_model)

LSTMClassifier(
  (embedding): Embedding(100278, 64, padding_idx=0)
  (lstm): LSTM(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [26]:
loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

In [27]:
epochs = 5

In [28]:
for epoch in range(epochs):

    lstm_model.train()

    optimizer.zero_grad()

    # Forward pass
    outputs = lstm_model(X_train_tensor)

    # Remove extra dimension
    outputs = outputs.squeeze()

    # Calculate loss
    loss = loss_function(
        outputs,
        y_train_tensor
    )

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    print(
        "Epoch:",
        epoch + 1,
        "Loss:",
        loss.item()
    )

Epoch: 1 Loss: 0.6953692436218262
Epoch: 2 Loss: 0.6939366459846497
Epoch: 3 Loss: 0.6925543546676636
Epoch: 4 Loss: 0.6912105679512024
Epoch: 5 Loss: 0.6898940801620483


In [29]:
lstm_model.eval()

with torch.no_grad():

    test_outputs = lstm_model(X_test_tensor)

    test_outputs = test_outputs.squeeze()

    predictions = (
        torch.sigmoid(test_outputs) >= 0.5
    ).float()

accuracy = (
    predictions == y_test_tensor
).float().mean()

print("LSTM Test Accuracy:", accuracy.item() * 100, "%")

LSTM Test Accuracy: 50.66666603088379 %


In [30]:
print("Number of new reviews:", len(new_reviews))

for review in new_reviews:
    print(review)

Number of new reviews: 3
This movie was absolutely fantastic. The acting and story were excellent.
I really disliked this movie. It was boring and the story was terrible.
The movie was enjoyable and had some great performances.


In [31]:
print("Number of new reviews:", len(new_reviews))

for review in new_reviews:
    print(review)

Number of new reviews: 3
This movie was absolutely fantastic. The acting and story were excellent.
I really disliked this movie. It was boring and the story was terrible.
The movie was enjoyable and had some great performances.


In [32]:
new_reviews_tokens = [
    tokenize_text(review)
    for review in new_reviews
]

new_reviews_padded = [
    pad_sequence(tokens, MAX_LEN)
    for tokens in new_reviews_tokens
]

new_reviews_tensor = torch.tensor(
    new_reviews_padded,
    dtype=torch.long
)

print("New reviews tensor shape:", new_reviews_tensor.shape)

New reviews tensor shape: torch.Size([3, 100])


In [33]:
lstm_model.eval()

with torch.no_grad():

    outputs = lstm_model(new_reviews_tensor)

    probabilities = torch.sigmoid(
        outputs
    ).squeeze()

for review, probability in zip(
    new_reviews,
    probabilities
):

    if probability >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    print("Review:", review)
    print("LSTM Predicted Sentiment:", sentiment)
    print("Probability:", probability.item())
    print("-" * 60)

Review: This movie was absolutely fantastic. The acting and story were excellent.
LSTM Predicted Sentiment: Negative
Probability: 0.4834953844547272
------------------------------------------------------------
Review: I really disliked this movie. It was boring and the story was terrible.
LSTM Predicted Sentiment: Negative
Probability: 0.4834953844547272
------------------------------------------------------------
Review: The movie was enjoyable and had some great performances.
LSTM Predicted Sentiment: Negative
Probability: 0.4834953844547272
------------------------------------------------------------


In [36]:
model.eval()
lstm_model.eval()

with torch.no_grad():

    # RNN accuracy
    rnn_outputs = model(X_test_tensor)
    rnn_outputs = rnn_outputs.squeeze()

    rnn_predictions = (
        torch.sigmoid(rnn_outputs) >= 0.5
    ).float()

    rnn_accuracy = (
        rnn_predictions == y_test_tensor
    ).float().mean()

    # LSTM accuracy
    lstm_outputs = lstm_model(X_test_tensor)
    lstm_outputs = lstm_outputs.squeeze()

    lstm_predictions = (
        torch.sigmoid(lstm_outputs) >= 0.5
    ).float()

    lstm_accuracy = (
        lstm_predictions == y_test_tensor
    ).float().mean()

print("RNN Accuracy :", rnn_accuracy.item() * 100, "%")
print("LSTM Accuracy:", lstm_accuracy.item() * 100, "%")

RNN Accuracy : 49.000000953674316 %
LSTM Accuracy: 50.66666603088379 %


In [37]:
print(df["sentiment"].value_counts())

sentiment
1    1509
0    1491
Name: count, dtype: int64


In [38]:
class RNNClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        embedded = self.embedding(x)

        output, hidden = self.rnn(embedded)

        # Use the final hidden state
        last_hidden = hidden[-1]

        output = self.fc(last_hidden)

        return output

In [39]:
vocab_size = tokenizer.n_vocab + 1

embedding_dim = 64
hidden_dim = 64

model = RNNClassifier(
    vocab_size,
    embedding_dim,
    hidden_dim
)

print(model)

RNNClassifier(
  (embedding): Embedding(100278, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [40]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [41]:
loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [42]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch_X, batch_y in train_loader:

        optimizer.zero_grad()

        outputs = model(batch_X).squeeze()

        loss = loss_function(
            outputs,
            batch_y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        "Epoch:",
        epoch + 1,
        "Loss:",
        average_loss
    )

Epoch: 1 Loss: 0.6973380756378174
Epoch: 2 Loss: 0.671671314239502
Epoch: 3 Loss: 0.6454114917914072
Epoch: 4 Loss: 0.6025850121180216
Epoch: 5 Loss: 0.5579709275563558
Epoch: 6 Loss: 0.48323443373044334
Epoch: 7 Loss: 0.46331140637397766
Epoch: 8 Loss: 0.3940860752264659
Epoch: 9 Loss: 0.3238769861062368
Epoch: 10 Loss: 0.24518718163172404


In [43]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        outputs = model(batch_X).squeeze()

        predictions = (
            torch.sigmoid(outputs) >= 0.5
        ).float()

        correct += (
            predictions == batch_y
        ).sum().item()

        total += batch_y.size(0)

rnn_accuracy = correct / total * 100

print("Improved RNN Accuracy:", rnn_accuracy, "%")

Improved RNN Accuracy: 50.33333333333333 %


In [44]:
class LSTMClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(embedded)

        # Use the final hidden state
        last_hidden = hidden[-1]

        output = self.fc(last_hidden)

        return output

In [45]:
lstm_model = LSTMClassifier(
    vocab_size,
    embedding_dim,
    hidden_dim
)

print(lstm_model)

LSTMClassifier(
  (embedding): Embedding(100278, 64, padding_idx=0)
  (lstm): LSTM(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [46]:
loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

In [47]:
epochs = 10

for epoch in range(epochs):

    lstm_model.train()

    total_loss = 0

    for batch_X, batch_y in train_loader:

        optimizer.zero_grad()

        outputs = lstm_model(batch_X).squeeze()

        loss = loss_function(
            outputs,
            batch_y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        "Epoch:",
        epoch + 1,
        "Loss:",
        average_loss
    )

Epoch: 1 Loss: 0.6930994510650634
Epoch: 2 Loss: 0.6734317390124003
Epoch: 3 Loss: 0.6370514019330342
Epoch: 4 Loss: 0.5690938444932302
Epoch: 5 Loss: 0.4747945471604665
Epoch: 6 Loss: 0.3583987655242284
Epoch: 7 Loss: 0.27665506581465404
Epoch: 8 Loss: 0.17833702862262726
Epoch: 9 Loss: 0.1420993988464276
Epoch: 10 Loss: 0.10450601495802403


In [48]:
lstm_model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        outputs = lstm_model(batch_X).squeeze()

        predictions = (
            torch.sigmoid(outputs) >= 0.5
        ).float()

        correct += (
            predictions == batch_y
        ).sum().item()

        total += batch_y.size(0)

lstm_accuracy = correct / total * 100

print("Improved LSTM Accuracy:", lstm_accuracy, "%")

Improved LSTM Accuracy: 57.666666666666664 %
